# 03 — Fine-tune & probe

Two ways to turn pragmatiq embeddings into a task model:
a fast **probe** (a classifier on frozen embeddings — gradient boosting by default,
with `logistic`/`lightgbm` selectable) and a **LoRA fine-tune** (frozen backbone plus
low-rank adapters plus a head). The probe is the cheapest signal check; LoRA is the
supervised path when you want to adapt the backbone to one task.

> pragmatiq is an independent implementation inspired by the PRAGMA paper
> (arXiv 2604.08649) and is not affiliated with or endorsed by Revolut.

## Setup

Notebook 03 reuses the tokenized book and pretrained run from notebook 02.

In [ ]:
from pathlib import Path

from pragmatiq import api

WORK = Path("runs_quickstart")
RUN_DIR = WORK / "runs" / "nb02"
LABELS = WORK / "raw" / "labels" / "default_12m.parquet"

## Probe vs raw-count baseline

The probe keeps the backbone frozen and fits a gradient-boosting classifier over user
embeddings, reporting **ROC-AUC and PR-AUC**. The raw-count baseline uses the *same*
classifier on hand-made event counts, so the gap is a clean read on representation
quality (the embedding vs trivial counts), not the choice of model.

In [ ]:
probe = api.probe(WORK / "tok", RUN_DIR, LABELS)  # probe_model="gbdt" by default

print(
    f"probe ROC-AUC {probe['probe_auc']:.3f} / PR-AUC {probe['probe_pr_auc']:.3f}\n"
    f"baseline ROC-AUC {probe['baseline_auc']:.3f} / PR-AUC {probe['baseline_pr_auc']:.3f}\n"
    f"(prevalence {probe['prevalence']:.3f}, head '{probe['probe_model']}')"
)

## LoRA fine-tune

LoRA injects rank-8 adapters into attention/FFN layers, then trains only the
adapters and a classification head on the `[USR]` embedding with early
stopping.

In [ ]:
finetune = api.finetune(
    WORK / "tok",
    RUN_DIR,
    LABELS,
    config={"max_epochs": 8, "lora_rank": 8},
)

print(
    f"best validation AUC {finetune['best_val_auc']:.3f} "
    f"({finetune['n_adapted']} LoRA adapters, {finetune['epochs_run']} epochs)"
)

## Which events drove a prediction?

Integrated gradients attribute a user's score to individual events. See
`pragmatiq.inference.explain.EventAttributor` and the Streamlit demo for the
interactive view.